In [1]:
# import key libraries
import numpy as np
import matplotlib.pyplot as plt
import requests

# PyTorch libraries
import torch
import torch.nn.functional as F

# Hugging Face libraries
from transformers import AutoTokenizer, AutoModelForCausalLM, GPT2Tokenizer
from datasets import load_dataset



In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [8]:
# Load small GPT2 pre-trained model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2 = AutoModelForCausalLM.from_pretrained('gpt2').to(device)
gpt2.eval();

In [18]:
# Function to ccalculate perplexity
def calculate_perplexity(tokens: torch.tensor,
                   model: torch.nn.Module=gpt2,
                   seq_length: int=1024,
                   device: str=device
                   ) -> float:

                   # find number of segments in the token sequence
                   num_segments = torch.numel(tokens) // seq_length

                   # Initialize losses
                   sum_losses = 0.

                   for i in range(num_segments):

                    start = i * seq_length
                    end = start + seq_length

                    # Extract a sequence of tokens and add batch dimension
                    X = tokens[0,start:end].unsqueeze(0).to(device)

                    # Forward pass
                    with torch.no_grad():
                        output = model(X, labels=X)

                    # Calculate the losses
                    sum_losses += output.loss.item()

                    # Calculate perplexity
                    perplexity = torch.exp(torch.tensor(sum_losses)/num_segments)

                    return perplexity

In [19]:
# generate random 50K tokens
rand_tokens = torch.randint(0, tokenizer.vocab_size, (1,50_000))

# test the perplexity
perplxity = calculate_perplexity(rand_tokens)
print(f'Perplexity: {perplxity:.4f}')

Perplexity: 1.2879
